# 35 — Causal Multi-Head Attention

**Master syllabus coverage:** V2 5.6–5.8

## Why this matters

Multi-head attention is where shape complexity sharply increases. A transparent implementation makes head splitting, masking, concatenation, projection, and residual integration auditable.

## Learning objectives

- Split combined projections into `[B,H,T,D]` heads and join them losslessly.
- Compute causal attention independently for every head.
- Project concatenated heads into a residual update.
- Package the operation in a readable PyTorch module.

Work through the notebook in order. Predict shapes and results before running each code cell. An assertion failure is part of the lesson: read it before changing code.


## 1. Multiple heads create multiple retrieval subspaces

Project to combined $Q,K,V$ with width $C$, reshape `C = H × D`, and transpose to
`[B,H,T,D]`. Each head produces `[B,H,T,D]`; concatenate back to `[B,T,C]`, then apply
output projection $W_O$. Heads do not permanently own fixed human-interpretable roles.


In [ ]:
import math
import torch

torch.manual_seed(42)
B, T, C, H = 2, 5, 12, 3
assert C % H == 0
D = C // H
x = torch.randn(B, T, C)
qkv_weight = torch.randn(C, 3 * C) / math.sqrt(C)
output_weight = torch.randn(C, C) / math.sqrt(C)

qkv = x @ qkv_weight                                  # [B,T,3C]
q, k, v = qkv.chunk(3, dim=-1)                        # each [B,T,C]

def split_heads(tensor):
    return tensor.view(B, T, H, D).transpose(1, 2)    # [B,H,T,D]

q, k, v = map(split_heads, (q, k, v))
scores = q @ k.transpose(-2, -1) / math.sqrt(D)       # [B,H,T,T]
allowed = torch.tril(torch.ones(T, T, dtype=torch.bool))
weights = torch.softmax(scores.masked_fill(~allowed, float("-inf")), dim=-1)
heads = weights @ v                                   # [B,H,T,D]
joined = heads.transpose(1, 2).contiguous().view(B, T, C)
update = joined @ output_weight                       # [B,T,C]
print("qkv/head/scores/update:", qkv.shape, q.shape, scores.shape, update.shape)


## 2. Reshape/transpose order is semantic

`view(B,T,H,D).transpose(1,2)` groups adjacent features into heads. A different reshape
can preserve element count yet assign values to the wrong head/time coordinates. After
transposition, call `contiguous()` before `view`; `reshape` may silently allocate.


In [ ]:
marker = torch.arange(B * T * C).reshape(B, T, C)
head_marker = marker.view(B, T, H, D).transpose(1, 2)
round_trip = head_marker.transpose(1, 2).contiguous().view(B, T, C)
assert torch.equal(marker, round_trip)
print("head 0, token 0 features:", head_marker[0, 0, 0].tolist())


## 3. Residual update and dropout

Multi-head attention returns a full-width update; the decoder adds it to the input
residual stream. Attention dropout acts on attention probabilities during training;
residual dropout acts on projected updates. Both must be disabled for deterministic
evaluation and generation.


In [ ]:
residual_output = x + update
assert residual_output.shape == x.shape
assert torch.isfinite(residual_output).all()
torch.testing.assert_close(weights.sum(dim=-1), torch.ones(B, H, T))
assert not weights.triu(diagonal=1).any()
print("causal and normalization invariants passed for every batch/head/query")


## 4. A reusable transparent module

This implementation deliberately keeps split, mask, softmax, join, and projection visible.
Later the production brain can replace its internals with fused SDPA after equivalence tests.


In [ ]:
class CausalSelfAttention(torch.nn.Module):
    def __init__(self, width: int, heads: int):
        super().__init__()
        assert width % heads == 0
        self.heads = heads
        self.head_dim = width // heads
        self.qkv = torch.nn.Linear(width, 3 * width, bias=False)
        self.out = torch.nn.Linear(width, width, bias=False)

    def forward(self, x):
        batch, time, width = x.shape
        q, k, v = self.qkv(x).chunk(3, dim=-1)
        def heads(value):
            return value.view(batch, time, self.heads, self.head_dim).transpose(1, 2)
        q, k, v = map(heads, (q, k, v))
        scores = q @ k.transpose(-2, -1) / math.sqrt(self.head_dim)
        allowed = torch.ones(time, time, dtype=torch.bool, device=x.device).tril()
        attention = torch.softmax(scores.masked_fill(~allowed, float("-inf")), dim=-1)
        values = attention @ v
        values = values.transpose(1, 2).contiguous().view(batch, time, width)
        return self.out(values), attention

module = CausalSelfAttention(C, H)
result, attention = module(x)
assert result.shape == x.shape and attention.shape == (B, H, T, T)


## Failure modes to recognize

- **Width not divisible by heads:** head dimension is not integral.
- **Incorrect reshape order:** features silently map to the wrong token/head.
- **Missing contiguous conversion:** view fails after transpose or data is unexpectedly copied elsewhere.
- **Head output not projected:** concatenated subspaces do not receive learned mixing before residual addition.

These are diagnostic patterns, not trivia. When a future model fails, reduce the problem to the smallest example in this notebook and test the relevant invariant.


## Deliberate practice

1. Add attention and residual dropout with correct train/eval behavior.
2. Copy module weights into an equivalent SDPA implementation and compare outputs and gradients.
3. Write assertions that detect future attention for every batch and head.

Do not merely rerun the provided cells. Make a prediction, change one variable, and write down why the result did or did not match your prediction.

## Exit ticket

**You are ready to continue when:** you can derive every multi-head shape, round-trip the layout, and prove causal probability invariants.

**Next:** 36 — Normalization, feed-forward networks, and a decoder block.
